In [ ]:
!pip install -q ultralytics roboflow

In [ ]:
from getpass import getpass
from roboflow import Roboflow

api_key = getpass("Enter your Roboflow API key: ")

rf = Roboflow(api_key=api_key)

project = rf.workspace("locatec").project("oranges-detection")

dataset = project.version(9).download("yolov11")

print("Dataset downloaded to:", dataset.location)

In [ ]:
import os
import yaml

data_yaml = os.path.join(dataset.location, "data.yaml")

with open(data_yaml, "r") as file:
    data_config = yaml.safe_load(file)

print("Class names:", data_config["names"])
print("\nNumber of classes:", data_config["nc"])
print("\nDataset paths:")
print("Train:", data_config.get("train"))
print("Validation:", data_config.get("val"))
print("Test:", data_config.get("test"))

In [ ]:
import os

for split in ["train", "valid", "test"]:
    image_dir = os.path.join(dataset.location, split, "images")

    if os.path.exists(image_dir):
        images = [
            f for f in os.listdir(image_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
        ]
        print(f"{split}: {len(images)} images")
        print("Example:", images[0] if images else "No images found")
    else:
        print(f"{split}: Folder not found at {image_dir}")

In [ ]:
import ultralytics

print("Ultralytics version:", ultralytics.__version__)

from ultralytics import YOLO

model = YOLO("yolo26n.pt")
print("YOLO26 Nano loaded successfully!")

In [ ]:

from ultralytics import YOLO

# Load the pretrained YOLO26 Nano model
model = YOLO("yolo26n.pt")

# Train on your Roboflow orange dataset
results = model.train(
    data=data_yaml,
    epochs=50,
    imgsz=640,
    batch=16,
    project="/content/orange_training",
    name="orange_detector",
    patience=10,
    plots=True,
    seed=42
)


In [ ]:

import os

for root, dirs, files in os.walk("/content/orange_training"):
    for file in files:
        if file in ["best.pt", "last.pt"]:
            print(os.path.join(root, file))


In [ ]:
from ultralytics import YOLO

model = YOLO("/content/orange_training/orange_detector/weights/best.pt")

In [ ]:

import os
import shutil
from google.colab import drive, files

# Make sure Google Drive is mounted
drive.mount('/content/drive')

# Source: trained model
source = "/content/orange_training/orange_detector/weights/best.pt"

# Destination: Google Drive
folder = "/content/drive/MyDrive/orange_project"
os.makedirs(folder, exist_ok=True)

destination = os.path.join(folder, "best.pt")
shutil.copy2(source, destination)

print("Model saved to Google Drive:", destination)

# Download model to your PC
files.download(destination)


In [ ]:

from google.colab import files

files.download("/content/drive/MyDrive/orange_project/best.pt")


In [ ]:

import glob
import os

# Find validation images using your dataset YAML path
print("data_yaml:", data_yaml)



In [ ]:

import glob
import os

# Find validation images
valid_dir = "/content/oranges-detection-9/valid/images"

image_paths = (
    glob.glob(os.path.join(valid_dir, "*.jpg")) +
    glob.glob(os.path.join(valid_dir, "*.jpeg")) +
    glob.glob(os.path.join(valid_dir, "*.png")) +
    glob.glob(os.path.join(valid_dir, "*.webp"))
)

print("Validation images found:", len(image_paths))
print("Example:", image_paths[0] if image_paths else "No images found")


In [ ]:

import random
from IPython.display import display
from PIL import Image

# Test 3 random validation images
sample_paths = random.sample(image_paths, min(3, len(image_paths)))

for path in sample_paths:
    results = model.predict(
        source=path,
        imgsz=960,
        conf=0.15,
        verbose=False
    )

    annotated = results[0].plot()

    print("Image:", os.path.basename(path))
    print("Detections:", len(results[0].boxes))
    display(Image.fromarray(annotated[:, :, ::-1]))


In [ ]:

from google.colab import files

uploaded = files.upload()


In [ ]:

import os

video_files = [
    name for name in uploaded.keys()
    if name.lower().endswith((".mp4", ".avi", ".mov", ".mkv"))
]

print("Uploaded video files:", video_files)


In [ ]:
import os
# print(os.path.exists("/content/Limes on Conveyor Belt at Citrus Processing Plant, Nature Stock Footage ft. limes & citrus.mp4"))
print(os.path.exists("/content/oranges.mp4"))

In [ ]:

import cv2
from IPython.display import display, Video

# video_path = "/content/oranges_input.mp4"
video_path = "/content/oranges.mp4"
output_path = "/content/oranges_detected3.mp4"

# Class IDs from your dataset
print("Class names:", model.names)

# Open video
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise ValueError("Could not open oranges.mp4")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Video: {width}x{height}, FPS: {fps:.2f}, Frames: {total_frames}")

# Write output video
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(
    output_path, fourcc, fps, (width, height)
)

frame_number = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model.predict(
        source=frame,
        imgsz=960,
        conf=0.15,
        verbose=False
    )

    # Draw predictions
    annotated = results[0].plot()
    writer.write(annotated)
    frame_number += 1

    if frame_number % 100 == 0:
        print(f"Processed {frame_number}/{total_frames} frames")

cap.release()
writer.release()

print("Finished! Output saved to:", output_path)


In [ ]:

display(Video(output_path, embed=True, width=800))


In [ ]:

from google.colab import files

files.download("/content/oranges_detected3.mp4")


In [ ]:

# import cv2
# from ultralytics import YOLO
# from IPython.display import display, Video

# model = YOLO("/content/orange_training/orange_detector-2/weights/best.pt")

# video_path = "/content/oranges.mp4"
# output_path = "/content/oranges_test_1280.mp4"

# cap = cv2.VideoCapture(video_path)

# fps = cap.get(cv2.CAP_PROP_FPS)
# width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
# height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# max_frames = int(fps * 10)  # First 10 seconds

# writer = cv2.VideoWriter(
#     output_path,
#     cv2.VideoWriter_fourcc(*"mp4v"),
#     fps,
#     (width, height)
# )

# frame_number = 0

# while frame_number < max_frames:
#     ret, frame = cap.read()
#     if not ret:
#         break

#     results = model.predict(
#         source=frame,
#         imgsz=1280,
#         conf=0.10,
#         verbose=False
#     )

#     writer.write(results[0].plot())
#     frame_number += 1

# cap.release()
# writer.release()

# print("Test completed:", output_path)


from ultralytics import YOLO
import cv2

model = YOLO("/content/orange_training/orange_detector/weights/best.pt")

video_path = "/content/oranges.mp4"
# output_path = "/content/oranges_tracking_compact.mp4"
output_path="/content/oranges_detected3.mp4"

cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

writer = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

for result in model.track(
    source=video_path,
    imgsz=1280,
    conf=0.10,
    tracker="bytetrack.yaml",
    classes=[1, 2],
    stream=True,
    persist=True,
    verbose=False
):
    frame = result.orig_img.copy()

    if result.boxes is not None and result.boxes.id is not None:
        boxes = result.boxes.xyxy.cpu().numpy().astype(int)
        ids = result.boxes.id.cpu().numpy().astype(int)

        for box, track_id in zip(boxes, ids):
            x1, y1, x2, y2 = box

            # Thin bounding box
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 1)

            # Compact ID label
            label = f"#{track_id}"
            font = cv2.FONT_HERSHEY_SIMPLEX
            font_scale = 0.4
            thickness = 1

            (tw, th), baseline = cv2.getTextSize(
                label, font, font_scale, thickness
            )

            label_y = max(y1, th + 4)
            cv2.rectangle(
                frame,
                (x1, label_y - th - 3),
                (x1 + tw + 4, label_y + baseline - 1),
                (0, 255, 0),
                -1
            )

            cv2.putText(
                frame,
                label,
                (x1 + 2, label_y - 2),
                font,
                font_scale,
                (0, 0, 0),
                thickness,
                cv2.LINE_AA
            )

    writer.write(frame)

cap.release()
writer.release()

print("Saved compact tracking video:", output_path)


In [ ]:
from google.colab import files

files.download("/content/oranges_detected3.mp4")

UPDATED 1.0

In [ ]:

import yaml
from pathlib import Path
import ultralytics

# Read the default ByteTrack configuration
default_path = (
    Path(ultralytics.__file__).parent
    / "cfg" / "trackers" / "bytetrack.yaml"
)

with open(default_path, "r") as f:
    tracker_config = yaml.safe_load(f)

# Adjust tracking settings
tracker_config["track_high_thresh"] = 0.15
tracker_config["track_low_thresh"] = 0.05
tracker_config["new_track_thresh"] = 0.10
tracker_config["track_buffer"] = 60

# Save as a NEW configuration file
custom_tracker = "/content/bytetrack_early_oranges.yaml"

with open(custom_tracker, "w") as f:
    yaml.safe_dump(tracker_config, f, sort_keys=False)

print("New tracker configuration created successfully!")


In [ ]:

from ultralytics import YOLO
import cv2
import os

# Load your trained model
model = YOLO("/content/drive/MyDrive/orange_project/best.pt")

# Use your current sample video
video_path = "/content/oranges.mp4"

# Save to a NEW output file for comparison
output_path = "/content/oranges_detected4.mp4"

if not os.path.exists(video_path):
    raise FileNotFoundError(f"Video not found: {video_path}")

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise ValueError("Could not open input video.")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

writer = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

if not writer.isOpened():
    cap.release()
    raise RuntimeError("Could not create output video.")

for result in model.track(
    source=video_path,
    imgsz=1280,
    conf=0.10,
    tracker="/content/bytetrack_early_oranges.yaml",
    classes=[1, 2],
    stream=True,
    persist=True,
    verbose=False
):
    frame = result.orig_img.copy()

    if result.boxes is not None and result.boxes.id is not None:
        boxes = result.boxes.xyxy.cpu().numpy().astype(int)
        ids = result.boxes.id.cpu().numpy().astype(int)

        for (x1, y1, x2, y2), track_id in zip(boxes, ids):
            cv2.rectangle(
                frame, (x1, y1), (x2, y2), (0, 255, 0), 1
            )

            label = f"#{track_id}"
            cv2.putText(
                frame,
                label,
                (x1 + 2, max(y1 - 4, 12)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.4,
                (0, 255, 0),
                1,
                cv2.LINE_AA
            )

    writer.write(frame)

cap.release()
writer.release()

print("New tracking video saved:", output_path)


In [ ]:
from google.colab import files

files.download("/content/oranges_detected4.mp4")

UPGRADE 1.1

In [ ]:

from ultralytics import YOLO
import cv2
from IPython.display import display
from PIL import Image

model = YOLO("/content/drive/MyDrive/orange_project/best.pt")
video_path = "/content/oranges.mp4"

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise ValueError("Could not open video.")

fps = cap.get(cv2.CAP_PROP_FPS)
print("FPS:", fps)

# Inspect frames at 1, 2, 3 and 4 seconds
for seconds in [1, 2, 3, 4]:
    cap.set(cv2.CAP_PROP_POS_MSEC, seconds * 1000)
    ret, frame = cap.read()

    if not ret:
        print(f"Could not read frame at {seconds}s")
        continue

    result = model.predict(
        source=frame,
        imgsz=1280,
        conf=0.10,
        classes=[1, 2],
        verbose=False
    )[0]

    print(f"Time: {seconds}s | Detections: {len(result.boxes)}")
    display(Image.fromarray(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)))

cap.release()


In [ ]:

from ultralytics import YOLO
import cv2
from IPython.display import display
from PIL import Image

model = YOLO("/content/drive/MyDrive/orange_project/best.pt")
video_path = "/content/oranges.mp4"

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise ValueError("Could not open video.")

# Inspect frames at 1, 2, 3 and 4 seconds
for seconds in [1, 2, 3, 4]:
    cap.set(cv2.CAP_PROP_POS_MSEC, seconds * 1000)
    ret, frame = cap.read()

    if not ret:
        print(f"Could not read frame at {seconds}s")
        continue

    result = model.predict(
        source=frame,
        imgsz=1280,
        conf=0.10,
        classes=[1, 2],
        verbose=False
    )[0]

    # Draw only thin green boxes; no labels or confidence scores
    clean_frame = frame.copy()

    if result.boxes is not None:
        boxes = result.boxes.xyxy.cpu().numpy().astype(int)

        for x1, y1, x2, y2 in boxes:
            cv2.rectangle(
                clean_frame, (x1, y1), (x2, y2), (0, 255, 0), 2
            )

    print(f"Time: {seconds}s | Detections: {len(result.boxes)}")
    display(Image.fromarray(cv2.cvtColor(clean_frame, cv2.COLOR_BGR2RGB)))

cap.release()


In [ ]:

from ultralytics import YOLO
import cv2

model = YOLO("/content/drive/MyDrive/orange_project/best.pt")
video_path = "/content/oranges.mp4"

cap = cv2.VideoCapture(video_path)

for seconds in [1, 2, 3, 4]:
    cap.set(cv2.CAP_PROP_POS_MSEC, seconds * 1000)
    ret, frame = cap.read()

    if not ret:
        print(f"Could not read frame at {seconds}s")
        continue

    print(f"\nFrame at {seconds} seconds:")

    for conf in [0.10, 0.05, 0.01]:
        result = model.predict(
            source=frame,
            imgsz=1280,
            conf=conf,
            classes=[1, 2],
            verbose=False
        )[0]

        print(f"  Confidence {conf:.2f}: {len(result.boxes)} detections")

cap.release()


In [ ]:

from ultralytics import YOLO
import cv2
from PIL import Image
from IPython.display import display

model = YOLO("/content/drive/MyDrive/orange_project/best.pt")
video_path = "/content/oranges.mp4"

cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_MSEC, 3000)
ret, frame = cap.read()
cap.release()

if ret:
    for conf in [0.10, 0.05]:
        result = model.predict(
            source=frame,
            imgsz=1280,
            conf=conf,
            classes=[1, 2],
            verbose=False
        )[0]

        output = frame.copy()

        for box in result.boxes.xyxy.cpu().numpy().astype(int):
            x1, y1, x2, y2 = box
            cv2.rectangle(output, (x1, y1), (x2, y2), (0, 255, 0), 2)

        print(f"Confidence: {conf} | Detections: {len(result.boxes)}")
        display(Image.fromarray(
            cv2.cvtColor(output, cv2.COLOR_BGR2RGB)
        ))
else:
    print("Could not read the video frame.")


In [ ]:

from ultralytics import YOLO
import cv2
from PIL import Image
from IPython.display import display

model = YOLO("/content/drive/MyDrive/orange_project/best.pt")
cap = cv2.VideoCapture("/content/oranges.mp4")
cap.set(cv2.CAP_PROP_POS_MSEC, 3000)
ret, frame = cap.read()
cap.release()

if ret:
    result = model.predict(
        source=frame,
        imgsz=1536,
        conf=0.10,
        classes=[1, 2],
        verbose=False
    )[0]

    output = frame.copy()

    for box in result.boxes.xyxy.cpu().numpy().astype(int):
        x1, y1, x2, y2 = box
        cv2.rectangle(output, (x1, y1), (x2, y2), (0, 255, 0), 2)

    print("Detections:", len(result.boxes))
    display(Image.fromarray(cv2.cvtColor(output, cv2.COLOR_BGR2RGB)))
else:
    print("Could not read the video frame.")


In [ ]:

from ultralytics import YOLO
import cv2

model = YOLO("/content/drive/MyDrive/orange_project/best.pt")
cap = cv2.VideoCapture("/content/oranges.mp4")
cap.set(cv2.CAP_PROP_POS_MSEC, 3000)
ret, frame = cap.read()
cap.release()

if ret:
    for size in [1536, 1920]:
        result = model.predict(
            source=frame,
            imgsz=size,
            conf=0.10,
            classes=[1, 2],
            verbose=False
        )[0]

        print(f"Image size: {size} | Detections: {len(result.boxes)}")
else:
    print("Could not read the video frame.")


In [ ]:

import cv2
from PIL import Image
from IPython.display import display

cap = cv2.VideoCapture("/content/oranges.mp4")
cap.set(cv2.CAP_PROP_POS_MSEC, 3000)
ret, frame = cap.read()
cap.release()

if ret:
    h, w = frame.shape[:2]

    # Crop the middle area; we will adjust it after seeing the result
    crop = frame[int(h * 0.20):int(h * 0.80),
                 int(w * 0.25):int(w * 0.75)]

    print("Original frame size:", w, "x", h)
    print("Crop size:", crop.shape[1], "x", crop.shape[0])
    display(Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)))
else:
    print("Could not read the video frame.")


In [ ]:

from ultralytics import YOLO
import cv2
from PIL import Image
from IPython.display import display

model = YOLO("/content/drive/MyDrive/orange_project/best.pt")

cap = cv2.VideoCapture("/content/oranges.mp4")
cap.set(cv2.CAP_PROP_POS_MSEC, 3000)
ret, frame = cap.read()
cap.release()

if ret:
    h, w = frame.shape[:2]

    x1, x2 = int(w * 0.25), int(w * 0.75)
    y1, y2 = int(h * 0.20), int(h * 0.80)
    crop = frame[y1:y2, x1:x2]

    result = model.predict(
        source=crop,
        imgsz=1280,
        conf=0.10,
        classes=[1, 2],
        verbose=False
    )[0]

    output = crop.copy()

    for box in result.boxes.xyxy.cpu().numpy().astype(int):
        bx1, by1, bx2, by2 = box
        cv2.rectangle(output, (bx1, by1), (bx2, by2), (0, 255, 0), 2)

    print("Detections in crop:", len(result.boxes))
    display(Image.fromarray(cv2.cvtColor(output, cv2.COLOR_BGR2RGB)))
else:
    print("Could not read the video frame.")


In [ ]:

from ultralytics import YOLO
import cv2
from IPython.display import Video, display

# Load your trained model
model = YOLO("/content/drive/MyDrive/orange_project/best.pt")

# Input and output paths
video_path = "/content/oranges.mp4"
output_path = "/content/oranges_tracking_1536.mp4"

# Open the video
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise ValueError(f"Could not open video: {video_path}")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Prepare output video
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(
    output_path, fourcc, fps, (width, height)
)

frame_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Detect and track oranges
    result = model.track(
        source=frame,
        imgsz=1536,
        conf=0.10,
        classes=[1, 2],
        tracker="bytetrack.yaml",
        persist=True,
        verbose=False
    )[0]

    # Draw thin green boxes and compact track IDs
    if result.boxes is not None and len(result.boxes) > 0:
        boxes = result.boxes.xyxy.cpu().numpy().astype(int)
        track_ids = (
            result.boxes.id.int().cpu().tolist()
            if result.boxes.id is not None
            else [None] * len(boxes)
        )

        for (x1, y1, x2, y2), track_id in zip(boxes, track_ids):
            cv2.rectangle(
                frame, (x1, y1), (x2, y2),
                (0, 255, 0), 2
            )

            if track_id is not None:
                cv2.putText(
                    frame, f"#{track_id}",
                    (x1, max(y1 - 5, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.45, (0, 255, 0), 1,
                    cv2.LINE_AA
                )

    writer.write(frame)
    frame_count += 1

    if frame_count % 100 == 0:
        print(f"Processed {frame_count} frames")

cap.release()
writer.release()

print(f"Finished! Total frames processed: {frame_count}")
print(f"Output saved to: {output_path}")


In [ ]:
from google.colab import files

files.download("/content/oranges_tracking_1536.mp4")

HORIZONTAL LINE

In [ ]:

from ultralytics import YOLO
import cv2
from IPython.display import Video, display

model = YOLO("/content/drive/MyDrive/orange_project/best.pt")

video_path = "/content/oranges.mp4"
output_path = "/content/oranges_with_line.mp4"

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise ValueError(f"Could not open video: {video_path}")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Horizontal counting line: adjust this value if needed
line_y = int(height * 0.30)

writer = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

frame_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    result = model.track(
        source=frame,
        imgsz=1536,
        conf=0.10,
        classes=[1, 2],
        tracker="bytetrack.yaml",
        persist=True,
        verbose=False
    )[0]

    # Draw thin green boxes and track IDs
    if result.boxes is not None and len(result.boxes) > 0:
        boxes = result.boxes.xyxy.cpu().numpy().astype(int)
        track_ids = (
            result.boxes.id.int().cpu().tolist()
            if result.boxes.id is not None
            else [None] * len(boxes)
        )

        for (x1, y1, x2, y2), track_id in zip(boxes, track_ids):
            cv2.rectangle(
                frame, (x1, y1), (x2, y2),
                (0, 255, 0), 2
            )

            if track_id is not None:
                cv2.putText(
                    frame, f"#{track_id}",
                    (x1, max(y1 - 5, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.45, (0, 255, 0), 1,
                    cv2.LINE_AA
                )

    # Draw the horizontal counting line in red
    cv2.line(
        frame, (0, line_y), (width, line_y),
        (0, 0, 255), 2
    )

    writer.write(frame)
    frame_count += 1

    if frame_count % 100 == 0:
        print(f"Processed {frame_count} frames")

cap.release()
writer.release()

print(f"Finished! Processed {frame_count} frames.")
print(f"Saved to: {output_path}")

display(Video(output_path, embed=True, width=800))


In [ ]:
from google.colab import files

files.download("/content/oranges_with_line.mp4")

VIDEO

In [ ]:

from ultralytics import YOLO
import cv2
import math
from IPython.display import Video, display

# -------------------- MODEL & VIDEO --------------------
model = YOLO("/content/drive/MyDrive/orange_project/best.pt")

video_path = "/content/oranges.mp4"
output_path = "/content/oranges_linkedin_final.mp4"

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise ValueError(f"Could not open video: {video_path}")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

if not fps or fps <= 0:
    fps = 30.0

line_y = int(height * 0.30)

writer = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

if not writer.isOpened():
    cap.release()
    raise RuntimeError("Could not create output video.")

# -------------------- COUNTING STATE --------------------
previous_y = {}
counted_ids = set()
total_count = 0
frame_count = 0

# Neon green, red and white in OpenCV BGR format
GREEN = (70, 255, 120)
RED = (50, 50, 255)
WHITE = (245, 250, 255)
DARK = (18, 25, 34)
CYAN = (220, 210, 50)


def draw_panel(frame, x1, y1, x2, y2,
               color=(18, 25, 34), alpha=0.78):
    """Draw a translucent dark panel."""
    overlay = frame.copy()
    cv2.rectangle(overlay, (x1, y1), (x2, y2), color, -1)
    cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0, frame)
    cv2.rectangle(frame, (x1, y1), (x2, y2), CYAN, 1)


def draw_corner_box(frame, x1, y1, x2, y2):
    """Draw a clean neon-style box with highlighted corners."""
    # Subtle glow
    glow = frame.copy()
    cv2.rectangle(glow, (x1, y1), (x2, y2), GREEN, 5)
    cv2.addWeighted(glow, 0.18, frame, 0.82, 0, frame)

    # Thin main outline
    cv2.rectangle(frame, (x1, y1), (x2, y2), GREEN, 2)

    # Brighter corner accents
    length = max(8, min(18, (x2 - x1) // 4, (y2 - y1) // 4))
    for x, y, dx, dy in [
        (x1, y1, 1, 1),
        (x2, y1, -1, 1),
        (x1, y2, 1, -1),
        (x2, y2, -1, -1)
    ]:
        cv2.line(frame, (x, y), (x + dx * length, y), GREEN, 3)
        cv2.line(frame, (x, y), (x, y + dy * length), GREEN, 3)


def draw_glow_line(frame, y):
    """Draw a glowing red counting line."""
    overlay = frame.copy()
    cv2.line(overlay, (0, y), (width - 1, y), RED, 10)
    cv2.addWeighted(overlay, 0.22, frame, 0.78, 0, frame)
    cv2.line(frame, (0, y), (width - 1, y), RED, 2)

    # Small marker at the left
    cv2.circle(frame, (22, y), 7, WHITE, 2)
    cv2.circle(frame, (22, y), 3, RED, -1)


def draw_hud(frame, count):
    """Draw the professional title and live total counter."""
    scale = max(0.55, min(width, height) / 900)
    title_size = max(0.55, scale * 0.9)
    small_size = max(0.42, scale * 0.65)

    # Top-left title and counter
    panel_w = min(int(width * 0.38), 440)
    panel_h = 112
    draw_panel(frame, 20, 20, 20 + panel_w, 20 + panel_h)

    cv2.putText(
        frame, "AI-POWERED ORANGE DETECTION",
        (35, 49), cv2.FONT_HERSHEY_SIMPLEX,
        small_size, CYAN, 1, cv2.LINE_AA
    )

    cv2.putText(
        frame, "TOTAL COUNT",
        (35, 80), cv2.FONT_HERSHEY_SIMPLEX,
        small_size, WHITE, 1, cv2.LINE_AA
    )

    cv2.putText(
        frame, str(count),
        (panel_w - 65, 99), cv2.FONT_HERSHEY_SIMPLEX,
        title_size * 1.25, GREEN, 2, cv2.LINE_AA
    )

    # Top-right model badge
    badge_w = min(int(width * 0.30), 310)
    bx = width - badge_w - 20
    draw_panel(frame, bx, 20, width - 20, 66)

    cv2.putText(
        frame, "YOLO26n + ByteTrack",
        (bx + 14, 53), cv2.FONT_HERSHEY_SIMPLEX,
        small_size, WHITE, 1, cv2.LINE_AA
    )

    # Bottom-left technology caption
    footer_y = height - 52
    draw_panel(frame, 20, footer_y - 12, min(width - 20, 300),
               height - 15, alpha=0.82)

    cv2.putText(
        frame, "REAL-TIME OBJECT TRACKING",
        (32, footer_y + 12), cv2.FONT_HERSHEY_SIMPLEX,
        small_size, WHITE, 1, cv2.LINE_AA
    )

    # Bottom-right live status
    status_w = 135
    sx = width - status_w - 20
    draw_panel(frame, sx, height - 53, width - 20, height - 15)

    cv2.circle(frame, (sx + 15, height - 34), 4, GREEN, -1)
    cv2.putText(
        frame, "TRACKING",
        (sx + 27, height - 29), cv2.FONT_HERSHEY_SIMPLEX,
        small_size, WHITE, 1, cv2.LINE_AA
    )


# -------------------- PROCESS ENTIRE VIDEO --------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    result = model.track(
        source=frame,
        imgsz=1536,
        conf=0.10,
        classes=[1, 2],
        tracker="bytetrack.yaml",
        persist=True,
        verbose=False
    )[0]

    # Draw detections and count line crossings
    if result.boxes is not None and len(result.boxes) > 0:
        boxes = result.boxes.xyxy.cpu().numpy().astype(int)

        if result.boxes.id is not None:
            ids = result.boxes.id.int().cpu().tolist()
        else:
            ids = [None] * len(boxes)

        for (x1, y1, x2, y2), track_id in zip(boxes, ids):
            x1 = max(0, min(width - 1, x1))
            x2 = max(0, min(width - 1, x2))
            y1 = max(0, min(height - 1, y1))
            y2 = max(0, min(height - 1, y2))

            draw_corner_box(frame, x1, y1, x2, y2)

            # Compact track-ID tag
            if track_id is not None:
                label = f"ID {track_id}"
                font = cv2.FONT_HERSHEY_SIMPLEX
                font_scale = max(0.45, min(width, height) / 1400)
                (tw, th), baseline = cv2.getTextSize(
                    label, font, font_scale, 1
                )

                label_top = max(0, y1 - th - 12)
                cv2.rectangle(
                    frame,
                    (x1, label_top),
                    (min(width - 1, x1 + tw + 12),
                     min(height - 1, label_top + th + baseline + 8)),
                    (18, 35, 28), -1
                )
                cv2.putText(
                    frame, label,
                    (x1 + 6, label_top + th + 3),
                    font, font_scale, GREEN, 1, cv2.LINE_AA
                )

                # Count a tracked orange only once when it crosses
                # from below the line to above it.
                center_y = (y1 + y2) // 2

                if track_id in previous_y:
                    old_y = previous_y[track_id]

                    crossed_upward = old_y > line_y and center_y <= line_y

                    if crossed_upward and track_id not in counted_ids:
                        counted_ids.add(track_id)
                        total_count += 1

                previous_y[track_id] = center_y

    # Draw the counting line and polished HUD
    draw_glow_line(frame, line_y)
    draw_hud(frame, total_count)

    writer.write(frame)
    frame_count += 1

    if frame_count % 100 == 0:
        print(
            f"Processed {frame_count} frames | "
            f"Oranges counted: {total_count}"
        )

cap.release()
writer.release()

print("\nVideo processing complete!")
print("Frames processed:", frame_count)
print("Total oranges counted:", total_count)
print("Output:", output_path)

display(Video(output_path, embed=True, width=900))


In [ ]:
from google.colab import files

files.download("/content/oranges_linkedin_final.mp4")

In [ ]:

import os

paths = [
    "/content/oranges.mp4",
    "/content/oranges_linkedin_final.mp4",
    "/content/drive/MyDrive/orange_project/best.pt"
]

for path in paths:
    print("FOUND   " if os.path.exists(path) else "MISSING ", path)


In [ ]:

from google.colab import drive
drive.mount('/content/drive')


In [ ]:

import os

folder = "/content/drive/MyDrive/orange_project"

print("Folder exists:", os.path.exists(folder))

if os.path.exists(folder):
    for root, dirs, files in os.walk(folder):
        for file in files:
            print(os.path.join(root, file))
else:
    print("orange_project folder was not found.")


In [ ]:

from google.colab import files

uploaded = files.upload()


In [ ]:

import os

print("Input video exists:", os.path.exists("/content/oranges.mp4"))
print("Model exists:", os.path.exists(
    "/content/drive/MyDrive/orange_project/best.pt"
))


In [ ]:
import os
import torch

print("Video files:")
for f in os.listdir("/content"):
    if f.lower().endswith((".mp4", ".mov", ".avi", ".mkv")):
        print(" -", f)

print("\nCUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:

from ultralytics import YOLO
import cv2
import math
from IPython.display import Video, display

# -------------------- MODEL & VIDEO --------------------
model = YOLO("/content/drive/MyDrive/orange_project/best.pt")

video_path = "/content/oranges.mp4"
output_path = "/content/oranges_linkedin_final.mp4"

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise ValueError(f"Could not open video: {video_path}")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

if not fps or fps <= 0:
    fps = 30.0

line_y = int(height * 0.30)

writer = cv2.VideoWriter(
    output_path,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)

if not writer.isOpened():
    cap.release()
    raise RuntimeError("Could not create output video.")

# -------------------- COUNTING STATE --------------------
previous_y = {}
counted_ids = set()
total_count = 0
frame_count = 0

# Neon green, red and white in OpenCV BGR format
GREEN = (70, 255, 120)
RED = (50, 50, 255)
WHITE = (245, 250, 255)
DARK = (18, 25, 34)
CYAN = (220, 210, 50)


def draw_panel(frame, x1, y1, x2, y2,
               color=(18, 25, 34), alpha=0.78):
    """Draw a translucent dark panel."""
    overlay = frame.copy()
    cv2.rectangle(overlay, (x1, y1), (x2, y2), color, -1)
    cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0, frame)
    cv2.rectangle(frame, (x1, y1), (x2, y2), CYAN, 1)


def draw_corner_box(frame, x1, y1, x2, y2):
    """Draw a clean neon-style box with highlighted corners."""
    # Subtle glow
    glow = frame.copy()
    cv2.rectangle(glow, (x1, y1), (x2, y2), GREEN, 5)
    cv2.addWeighted(glow, 0.18, frame, 0.82, 0, frame)

    # Thin main outline
    cv2.rectangle(frame, (x1, y1), (x2, y2), GREEN, 2)

    # Brighter corner accents
    length = max(8, min(18, (x2 - x1) // 4, (y2 - y1) // 4))
    for x, y, dx, dy in [
        (x1, y1, 1, 1),
        (x2, y1, -1, 1),
        (x1, y2, 1, -1),
        (x2, y2, -1, -1)
    ]:
        cv2.line(frame, (x, y), (x + dx * length, y), GREEN, 3)
        cv2.line(frame, (x, y), (x, y + dy * length), GREEN, 3)


def draw_glow_line(frame, y):
    """Draw a glowing red counting line."""
    overlay = frame.copy()
    cv2.line(overlay, (0, y), (width - 1, y), RED, 10)
    cv2.addWeighted(overlay, 0.22, frame, 0.78, 0, frame)
    cv2.line(frame, (0, y), (width - 1, y), RED, 2)

    # Small marker at the left
    cv2.circle(frame, (22, y), 7, WHITE, 2)
    cv2.circle(frame, (22, y), 3, RED, -1)


def draw_hud(frame, count):
    """Draw the professional title and live total counter."""
    scale = max(0.55, min(width, height) / 900)
    title_size = max(0.55, scale * 0.9)
    small_size = max(0.42, scale * 0.65)

    # Top-left title and counter
    panel_w = min(int(width * 0.38), 440)
    panel_h = 112
    draw_panel(frame, 20, 20, 20 + panel_w, 20 + panel_h)

    cv2.putText(
        frame, "AI-POWERED ORANGE DETECTION",
        (35, 49), cv2.FONT_HERSHEY_SIMPLEX,
        small_size, CYAN, 1, cv2.LINE_AA
    )

    cv2.putText(
        frame, "TOTAL COUNT",
        (35, 80), cv2.FONT_HERSHEY_SIMPLEX,
        small_size, WHITE, 1, cv2.LINE_AA
    )

    cv2.putText(
        frame, str(count),
        (panel_w - 65, 99), cv2.FONT_HERSHEY_SIMPLEX,
        title_size * 1.25, GREEN, 2, cv2.LINE_AA
    )

    # Top-right model badge
    badge_w = min(int(width * 0.30), 310)
    bx = width - badge_w - 20
    draw_panel(frame, bx, 20, width - 20, 66)

    cv2.putText(
        frame, "YOLO26n + ByteTrack",
        (bx + 14, 53), cv2.FONT_HERSHEY_SIMPLEX,
        small_size, WHITE, 1, cv2.LINE_AA
    )

    # Bottom-left technology caption
    footer_y = height - 52
    draw_panel(frame, 20, footer_y - 12, min(width - 20, 300),
               height - 15, alpha=0.82)

    cv2.putText(
        frame, "REAL-TIME OBJECT TRACKING",
        (32, footer_y + 12), cv2.FONT_HERSHEY_SIMPLEX,
        small_size, WHITE, 1, cv2.LINE_AA
    )

    # Bottom-right live status
    status_w = 135
    sx = width - status_w - 20
    draw_panel(frame, sx, height - 53, width - 20, height - 15)

    cv2.circle(frame, (sx + 15, height - 34), 4, GREEN, -1)
    cv2.putText(
        frame, "TRACKING",
        (sx + 27, height - 29), cv2.FONT_HERSHEY_SIMPLEX,
        small_size, WHITE, 1, cv2.LINE_AA
    )


# -------------------- PROCESS ENTIRE VIDEO --------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    result = model.track(
        source=frame,
        imgsz=1536,
        conf=0.10,
        classes=[1, 2],
        tracker="bytetrack.yaml",
        persist=True,
        verbose=False
    )[0]

    # Draw detections and count line crossings
    if result.boxes is not None and len(result.boxes) > 0:
        boxes = result.boxes.xyxy.cpu().numpy().astype(int)

        if result.boxes.id is not None:
            ids = result.boxes.id.int().cpu().tolist()
        else:
            ids = [None] * len(boxes)

        for (x1, y1, x2, y2), track_id in zip(boxes, ids):
            x1 = max(0, min(width - 1, x1))
            x2 = max(0, min(width - 1, x2))
            y1 = max(0, min(height - 1, y1))
            y2 = max(0, min(height - 1, y2))

            draw_corner_box(frame, x1, y1, x2, y2)

            # Compact track-ID tag
            if track_id is not None:
                label = f"ID {track_id}"
                font = cv2.FONT_HERSHEY_SIMPLEX
                font_scale = max(0.45, min(width, height) / 1400)
                (tw, th), baseline = cv2.getTextSize(
                    label, font, font_scale, 1
                )

                label_top = max(0, y1 - th - 12)
                cv2.rectangle(
                    frame,
                    (x1, label_top),
                    (min(width - 1, x1 + tw + 12),
                     min(height - 1, label_top + th + baseline + 8)),
                    (18, 35, 28), -1
                )
                cv2.putText(
                    frame, label,
                    (x1 + 6, label_top + th + 3),
                    font, font_scale, GREEN, 1, cv2.LINE_AA
                )

                # Count a tracked orange only once when it crosses
                # from below the line to above it.
                center_y = (y1 + y2) // 2

                if track_id in previous_y:
                    old_y = previous_y[track_id]

                    crossed_upward = old_y > line_y and center_y <= line_y

                    if crossed_upward and track_id not in counted_ids:
                        counted_ids.add(track_id)
                        total_count += 1

                previous_y[track_id] = center_y

    # Draw the counting line and polished HUD
    draw_glow_line(frame, line_y)
    draw_hud(frame, total_count)

    writer.write(frame)
    frame_count += 1

    if frame_count % 100 == 0:
        print(
            f"Processed {frame_count} frames | "
            f"Oranges counted: {total_count}"
        )

cap.release()
writer.release()

print("\nVideo processing complete!")
print("Frames processed:", frame_count)
print("Total oranges counted:", total_count)
print("Output:", output_path)

display(Video(output_path, embed=True, width=900))


In [ ]:
from google.colab import files

files.download("/content/oranges_linkedin_final.mp4")

NEW INTERFACE

In [ ]:
from ultralytics import YOLO
import cv2
import math
import numpy as np
from IPython.display import Video, display

# -------------------- MODEL & VIDEO --------------------
model = YOLO("/content/drive/MyDrive/orange_project/best.pt")

video_path = "/content/oranges.mp4"
output_path = "/content/oranges_linkedin_final.mp4"

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise ValueError(f"Could not open video: {video_path}")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
if not fps or fps <= 0:
    fps = 30.0

line_y = int(height * 0.30)

writer = cv2.VideoWriter(
    output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height)
)
if not writer.isOpened():
    cap.release()
    raise RuntimeError("Could not create output video.")

# -------------------- COUNTING STATE --------------------
previous_y = {}
counted_ids = set()
flash_until = {}          # track_id -> frame number until which the box flashes white
rings = []                # burst rings created on each count
popups = []               # floating "+1" texts
total_count = 0
frame_count = 0
last_count_frame = -999

# -------------------- STYLE --------------------
# BGR colors
WHITE = (255, 255, 255)
RED = (60, 60, 255)
CYAN = (255, 230, 60)
LIME = (90, 255, 140)
PANEL = (16, 12, 22)
FONT = cv2.FONT_HERSHEY_SIMPLEX

UI_SCALE = max(0.6, min(width, height) / 900)


def neon_color(track_id):
    """Vibrant, evenly spread neon color per track ID (golden-angle hue)."""
    hue = int((track_id * 47) % 180)
    bgr = cv2.cvtColor(np.uint8([[[hue, 255, 255]]]), cv2.COLOR_HSV2BGR)[0][0]
    return int(bgr[0]), int(bgr[1]), int(bgr[2])


def draw_panel(frame, x1, y1, x2, y2, alpha=0.72, accent=CYAN):
    """Translucent dark panel with a glowing accent edge."""
    roi = frame[y1:y2, x1:x2]
    dark = np.full_like(roi, PANEL)
    frame[y1:y2, x1:x2] = cv2.addWeighted(dark, alpha, roi, 1 - alpha, 0)
    cv2.rectangle(frame, (x1, y1), (x2, y2), accent, 1, cv2.LINE_AA)
    # bright corner accents
    c = 14
    for x, y, dx, dy in [(x1, y1, 1, 1), (x2, y1, -1, 1),
                         (x1, y2, 1, -1), (x2, y2, -1, -1)]:
        cv2.line(frame, (x, y), (x + dx * c, y), WHITE, 2, cv2.LINE_AA)
        cv2.line(frame, (x, y), (x, y + dy * c), WHITE, 2, cv2.LINE_AA)


def draw_text_outline(frame, text, org, scale, color, thick=1):
    """Text with a dark outline so tiny labels stay readable on busy footage."""
    cv2.putText(frame, text, org, FONT, scale, (0, 0, 0),
                thick + 2, cv2.LINE_AA)
    cv2.putText(frame, text, org, FONT, scale, color, thick, cv2.LINE_AA)


def draw_counting_line(frame, y, t):
    """Pulsing glow line with a bright scanner sweeping along it."""
    pulse = 0.5 + 0.5 * math.sin(t * 0.15)

    glow = frame.copy()
    cv2.line(glow, (0, y), (width - 1, y), RED, int(14 + 6 * pulse))
    cv2.addWeighted(glow, 0.20 + 0.10 * pulse, frame, 0.80 - 0.10 * pulse,
                    0, frame)
    cv2.line(frame, (0, y), (width - 1, y), RED, 2, cv2.LINE_AA)

    # scanner sweep
    sweep_len = int(width * 0.14)
    sx = int((t * 14) % (width + sweep_len)) - sweep_len
    for i in range(sweep_len):
        x = sx + i
        if 0 <= x < width:
            k = i / sweep_len
            col = (int(60 + 195 * k), int(60 + 195 * k), 255)
            frame[max(0, y - 1):y + 2, x] = col

    # tick marks along the line
    for x in range(0, width, 60):
        cv2.line(frame, (x, y - 5), (x, y + 5), RED, 1, cv2.LINE_AA)

    # left marker
    cv2.circle(frame, (22, y), 8, WHITE, 1, cv2.LINE_AA)
    cv2.circle(frame, (22, y), 3, RED, -1, cv2.LINE_AA)


def draw_hud(frame, count, t):
    """Counter in the TOP-RIGHT corner, with a pop animation on each count."""
    margin = 20
    pw = int(min(width * 0.26, 260))
    ph = int(96 * UI_SCALE)
    x2 = width - margin
    x1 = x2 - pw
    y1 = margin
    y2 = y1 + ph
    draw_panel(frame, x1, y1, x2, y2)

    small = 0.45 * UI_SCALE
    draw_text_outline(frame, "ORANGES COUNTED",
                      (x1 + 12, y1 + int(22 * UI_SCALE)), small, CYAN)

    # pop animation: number scales up and flashes white right after a count
    age = t - last_count_frame
    pop = max(0.0, 1.0 - age / 12.0)
    scale = 1.5 * UI_SCALE * (1 + 0.35 * pop)
    color = WHITE if pop > 0.4 else LIME
    text = str(count)
    (tw, th), _ = cv2.getTextSize(text, FONT, scale, 3)
    tx = x2 - 14 - tw
    ty = y2 - int(16 * UI_SCALE)
    cv2.putText(frame, text, (tx, ty), FONT, scale, (0, 0, 0), 6, cv2.LINE_AA)
    cv2.putText(frame, text, (tx, ty), FONT, scale, color, 3, cv2.LINE_AA)

    # small live dot + model tag on the left of the number
    blink = 0.5 + 0.5 * math.sin(t * 0.25)
    cv2.circle(frame, (x1 + 18, y2 - int(30 * UI_SCALE)), 4,
               (int(60 + 120 * blink), 255, int(80 + 100 * blink)),
               -1, cv2.LINE_AA)
    draw_text_outline(frame, "LIVE", (x1 + 28, y2 - int(25 * UI_SCALE)),
                      small * 0.9, WHITE)
    draw_text_outline(frame, "YOLO26n + ByteTrack",
                      (x1 + 12, y2 - int(8 * UI_SCALE)), small * 0.7,
                      (200, 200, 200))


def draw_effects(frame, t):
    """Burst rings and floating +1 texts created when an orange is counted."""
    for r in rings[:]:
        a = t - r["start"]
        if a > 16:
            rings.remove(r)
            continue
        fade = 1 - a / 16
        col = tuple(int(c * fade + 255 * (1 - fade) * 0.0) for c in LIME)
        cv2.circle(frame, (r["x"], r["y"]), int(6 + a * 3.2), col, 2,
                   cv2.LINE_AA)
        cv2.circle(frame, (r["x"], r["y"]), int(3 + a * 1.8),
                   (int(255 * fade),) * 3, 1, cv2.LINE_AA)

    for p in popups[:]:
        a = t - p["start"]
        if a > 20:
            popups.remove(p)
            continue
        fade = 1 - a / 20
        col = tuple(int(c * fade) for c in LIME)
        draw_text_outline(frame, "+1", (p["x"] - 10, p["y"] - int(a * 1.8)),
                          0.55 * UI_SCALE, col, 1)


# -------------------- PROCESS ENTIRE VIDEO --------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    result = model.track(
        source=frame,
        imgsz=1536,
        conf=0.10,
        classes=[1, 2],
        tracker="bytetrack.yaml",
        persist=True,
        verbose=False
    )[0]

    tracked = []   # (x1, y1, x2, y2, id, color)

    if result.boxes is not None and len(result.boxes) > 0:
        boxes = result.boxes.xyxy.cpu().numpy().astype(int)
        if result.boxes.id is not None:
            ids = result.boxes.id.int().cpu().tolist()
        else:
            ids = [None] * len(boxes)

        for (x1, y1, x2, y2), track_id in zip(boxes, ids):
            x1 = int(max(0, min(width - 1, x1)))
            x2 = int(max(0, min(width - 1, x2)))
            y1 = int(max(0, min(height - 1, y1)))
            y2 = int(max(0, min(height - 1, y2)))

            if track_id is None:
                color = LIME
            elif flash_until.get(track_id, -1) >= frame_count:
                color = WHITE                      # white-hot flash on count
            else:
                color = neon_color(track_id)

            tracked.append((x1, y1, x2, y2, track_id, color))

            # ---- counting logic (unchanged): bottom -> top crossing ----
            if track_id is not None:
                center_y = (y1 + y2) // 2
                if track_id in previous_y:
                    old_y = previous_y[track_id]
                    crossed_upward = old_y > line_y and center_y <= line_y
                    if crossed_upward and track_id not in counted_ids:
                        counted_ids.add(track_id)
                        total_count += 1
                        last_count_frame = frame_count
                        flash_until[track_id] = frame_count + 8
                        cx = (x1 + x2) // 2
                        rings.append({"x": cx, "y": line_y,
                                      "start": frame_count})
                        popups.append({"x": cx, "y": line_y - 8,
                                       "start": frame_count})
                previous_y[track_id] = center_y

    # ---- boxes: ONE shared glow layer (fast even with crowded scenes) ----
    if tracked:
        glow = frame.copy()
        for x1, y1, x2, y2, _, color in tracked:
            cv2.rectangle(glow, (x1, y1), (x2, y2), color, 4)
        cv2.addWeighted(glow, 0.30, frame, 0.70, 0, frame)

        for x1, y1, x2, y2, track_id, color in tracked:
            # thin 1px outline
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 1, cv2.LINE_AA)

            # tiny corner accents
            L = max(4, min(8, (x2 - x1) // 4, (y2 - y1) // 4))
            for x, y, dx, dy in [(x1, y1, 1, 1), (x2, y1, -1, 1),
                                 (x1, y2, 1, -1), (x2, y2, -1, -1)]:
                cv2.line(frame, (x, y), (x + dx * L, y), WHITE, 1, cv2.LINE_AA)
                cv2.line(frame, (x, y), (x, y + dy * L), WHITE, 1, cv2.LINE_AA)

            # short label: just the ID number, no background box
            if track_id is not None:
                draw_text_outline(frame, str(track_id),
                                  (x1 + 2, max(10, y1 - 3)),
                                  0.34 * UI_SCALE, color, 1)

    draw_counting_line(frame, line_y, frame_count)
    draw_effects(frame, frame_count)
    draw_hud(frame, total_count, frame_count)

    writer.write(frame)
    frame_count += 1

    if frame_count % 100 == 0:
        print(f"Processed {frame_count} frames | "
              f"Oranges counted: {total_count}")

cap.release()
writer.release()

print("\nVideo processing complete!")
print("Frames processed:", frame_count)
print("Total oranges counted:", total_count)
print("Output:", output_path)

display(Video(output_path, embed=True, width=900))

In [ ]:
from google.colab import files

files.download("/content/oranges_linkedin_final.mp4")

OTHER INTERFACE

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
from IPython.display import Video, display

# -------------------- MODEL & VIDEO --------------------
model = YOLO("/content/drive/MyDrive/orange_project/best.pt")

video_path = "/content/oranges.mp4"
output_path = "/content/oranges_linkedin_final.mp4"

cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise ValueError(f"Could not open video: {video_path}")

fps = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
if not fps or fps <= 0:
    fps = 30.0

line_y = int(height * 0.30)

writer = cv2.VideoWriter(
    output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height)
)
if not writer.isOpened():
    cap.release()
    raise RuntimeError("Could not create output video.")

# -------------------- STATE --------------------
previous_y = {}
counted_ids = set()
total_count = 0
frame_count = 0
last_count_frame = -999

# -------------------- STYLE (BGR) --------------------
GREEN = (60, 255, 90)        # all boxes
BLACK = (0, 0, 0)
WHITE = (255, 255, 255)
LINE_COLOR = (255, 200, 80)  # calm light-blue counting line
PANEL = (20, 18, 24)
FONT = cv2.FONT_HERSHEY_SIMPLEX
UI = max(0.6, min(width, height) / 900)


def draw_line(frame, y):
    """Simple, decent single-color counting line."""
    cv2.line(frame, (0, y), (width - 1, y), LINE_COLOR, 2, cv2.LINE_AA)


def draw_counter(frame, count, t):
    """Counter panel in the top-right corner."""
    pw, ph = int(min(width * 0.24, 230)), int(80 * UI)
    x2, y1 = width - 20, 20
    x1, y2 = x2 - pw, y1 + ph

    roi = frame[y1:y2, x1:x2]
    frame[y1:y2, x1:x2] = cv2.addWeighted(
        np.full_like(roi, PANEL), 0.75, roi, 0.25, 0)
    cv2.rectangle(frame, (x1, y1), (x2, y2), GREEN, 1, cv2.LINE_AA)

    cv2.putText(frame, "ORANGES COUNTED", (x1 + 12, y1 + int(22 * UI)),
                FONT, 0.42 * UI, WHITE, 1, cv2.LINE_AA)

    pop = max(0.0, 1 - (t - last_count_frame) / 10)   # small pop on each count
    scale = 1.4 * UI * (1 + 0.25 * pop)
    text = str(count)
    (tw, _), _ = cv2.getTextSize(text, FONT, scale, 3)
    cv2.putText(frame, text, (x2 - 14 - tw, y2 - int(14 * UI)),
                FONT, scale, GREEN, 3, cv2.LINE_AA)


# -------------------- PROCESS VIDEO --------------------
while True:
    ret, frame = cap.read()
    if not ret:
        break

    result = model.track(
        source=frame, imgsz=1536, conf=0.10, classes=[1, 2],
        tracker="bytetrack.yaml", persist=True, verbose=False
    )[0]

    if result.boxes is not None and len(result.boxes) > 0:
        boxes = result.boxes.xyxy.cpu().numpy().astype(int)
        ids = (result.boxes.id.int().cpu().tolist()
               if result.boxes.id is not None else [None] * len(boxes))

        for (x1, y1, x2, y2), tid in zip(boxes, ids):
            x1 = int(max(0, min(width - 1, x1)))
            x2 = int(max(0, min(width - 1, x2)))
            y1 = int(max(0, min(height - 1, y1)))
            y2 = int(max(0, min(height - 1, y2)))

            # thin green box
            cv2.rectangle(frame, (x1, y1), (x2, y2), GREEN, 1, cv2.LINE_AA)

            if tid is not None:
                # mini green tag with the ID
                label = str(tid)
                fs = 0.3 * UI
                (tw, th), _ = cv2.getTextSize(label, FONT, fs, 1)
                ty1 = max(0, y1 - th - 4)
                cv2.rectangle(frame, (x1, ty1), (x1 + tw + 4, ty1 + th + 4),
                              GREEN, -1)
                cv2.putText(frame, label, (x1 + 2, ty1 + th + 1),
                            FONT, fs, BLACK, 1, cv2.LINE_AA)

                # count once when crossing the line bottom -> top
                cy = (y1 + y2) // 2
                if tid in previous_y:
                    if (previous_y[tid] > line_y and cy <= line_y
                            and tid not in counted_ids):
                        counted_ids.add(tid)
                        total_count += 1
                        last_count_frame = frame_count
                previous_y[tid] = cy

    draw_line(frame, line_y)
    draw_counter(frame, total_count, frame_count)

    writer.write(frame)
    frame_count += 1
    if frame_count % 100 == 0:
        print(f"Processed {frame_count} frames | Oranges counted: {total_count}")

cap.release()
writer.release()

print("\nVideo processing complete!")
print("Frames processed:", frame_count)
print("Total oranges counted:", total_count)
print("Output:", output_path)

display(Video(output_path, embed=True, width=900))

In [ ]:
from google.colab import files

files.download("/content/oranges_linkedin_final.mp4")